In [26]:
# pip install scikit-learn

In [27]:
# pip install pandas

**Import des données**

In [28]:
# import os
# import io
# #quelques variables globales utiles
# PATH_TO_HAM_DIR = "C:\\Users\\User\\OneDrive\\Documents\\Python\\WORKSHOP_B4\\emails\\ham"
# PATH_TO_SPAM_DIR = "C:\\Users\\User\\OneDrive\\Documents\\Python\\WORKSHOP_B4\\emails\\spam"
 
# SPAM_TYPE = "SPAM"
# HAM_TYPE = "HAM"
 
# #les tableaux X et Y seront ordonnés et de la même taille
# # X représente l'input Data (ici les mails)
# X = []
# #indique s'il s'agit d'un mail ou non
# Y = [] #les etiquettes (labels) pour le training set
 
 
# def readFilesFromDirectory(path, classification):
#     os.chdir(path)
#     files_name = os.listdir(path)
#     for current_file in files_name:
#         message = extract_mail_body(current_file)
#         X.append(message)
#         Y.append(classification)
        
            
 
# def extract_mail_body(file_name_str):
#     inBody = False
#     lines = []
#     file_descriptor = io.open(file_name_str,'r', encoding='latin1')
#     for line in file_descriptor:
#         if inBody:
#             lines.append(line)
#         elif line == '\n':
#             inBody = True
#         message = '\n'.join(lines)
#     file_descriptor.close()
#     return message
 
 
# readFilesFromDirectory(PATH_TO_HAM_DIR, HAM_TYPE)
# readFilesFromDirectory(PATH_TO_SPAM_DIR, SPAM_TYPE)

In [37]:
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.model_selection import train_test_split
# import pandas as pd  
# import numpy as np
# vectorizer = CountVectorizer()
 
# counts = vectorizer.fit_transform(training_set["X"].values)

**Correction de Gemini car ça marchait pas**

In [38]:
import os
import io
import pandas as pd # Import nécessaire
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

# --- 1. CONFIGURATION ---
PATH_TO_HAM_DIR = "C:\\Users\\User\\OneDrive\\Documents\\Python\\WORKSHOP_B4\\emails\\ham"
PATH_TO_SPAM_DIR = "C:\\Users\\User\\OneDrive\\Documents\\Python\\WORKSHOP_B4\\emails\\spam"

SPAM_TYPE = "SPAM"
HAM_TYPE = "HAM"

# Les listes qui vont recevoir les données
X = [] # Le contenu des emails
Y = [] # Les étiquettes (HAM ou SPAM)

# --- 2. FONCTIONS DE LECTURE ---
def extract_mail_body(file_name_str):
    inBody = False
    lines = []
    # On utilise 'latin1' pour éviter les erreurs de lecture
    file_descriptor = io.open(file_name_str, 'r', encoding='latin1')
    for line in file_descriptor:
        if inBody:
            lines.append(line)
        elif line == '\n':
            inBody = True
    file_descriptor.close()
    return '\n'.join(lines)

def readFilesFromDirectory(path, classification):
    # On sauvegarde le dossier actuel pour pouvoir y revenir plus tard
    original_path = os.getcwd() 
    os.chdir(path)
    
    files_name = os.listdir(path)
    print(f"Lecture de {path}...")
    
    for current_file in files_name:
        # On ignore les fichiers systèmes cachés s'il y en a
        if os.path.isfile(current_file):
            message = extract_mail_body(current_file)
            X.append(message)
            Y.append(classification)
            
    # IMPORTANT : On revient au dossier de départ pour ne pas se perdre
    os.chdir(original_path)

# --- 3. EXÉCUTION ---

# Remplissage des listes X et Y
readFilesFromDirectory(PATH_TO_HAM_DIR, HAM_TYPE)
readFilesFromDirectory(PATH_TO_SPAM_DIR, SPAM_TYPE)

# --- C'EST ICI LA CORRECTION ---
# On transforme vos listes X et Y en un DataFrame Pandas nommé 'training_set'
data = {'X': X, 'y': Y}
training_set = pd.DataFrame(data)

print(f"Données chargées. Total emails : {len(training_set)}")

# --- 4. MACHINE LEARNING ---
vectorizer = CountVectorizer(stop_words='english')

# Maintenant cette ligne fonctionne car training_set existe !
counts = vectorizer.fit_transform(training_set["X"].values)

print("Vectorization terminée !")
print(f"Taille de la matrice : {counts.shape}")

Lecture de C:\Users\User\OneDrive\Documents\Python\WORKSHOP_B4\emails\ham...
Lecture de C:\Users\User\OneDrive\Documents\Python\WORKSHOP_B4\emails\spam...
Données chargées. Total emails : 3000
Vectorization terminée !
Taille de la matrice : (3000, 62660)


In [39]:
from sklearn.naive_bayes import MultinomialNB
classifier = MultinomialNB()
targets = training_set['y'].values
classifier.fit(counts, targets)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


**TestduprogrammeOK**

In [41]:
examples = ['Free Viagra now!!!', "Hi Bob, how about a game of golf tomorrow?"]
 
example_counts = vectorizer.transform(examples)
 
predictions = classifier.predict(example_counts)

print(predictions)

['SPAM' 'HAM']


**Test du programme**

**Raté**

In [42]:
# ==========================================
# ENGLISH TEST ZONE
# ==========================================

print("\n--- STARTING ENGLISH TEST ---")

# 1. On définit des emails en ANGLAIS
# Le vocabulaire doit correspondre à ce que l'IA a appris (dataset Enron)
english_test_emails = [
    "Hi, can we reschedule our meeting to next Monday? Let me know. Thanks.", 
    "CONGRATULATIONS! You have won a $1,000 Walmart Gift Card. Click here to claim it now for FREE!"
    ""
]

# 2. On transforme ces phrases en vecteurs (chiffres)
# L'IA va chercher les mots qu'elle connait (meeting, monday, win, free, click...)
test_counts = vectorizer.transform(english_test_emails)

# 3. Prédiction
predictions = classifier.predict(test_counts)

# 4. Affichage des résultats
for i in range(len(english_test_emails)):
    print(f"\nEmail: {english_test_emails[i]}")
    print(f"--> AI Prediction: {predictions[i]}")

print("\n--- END OF TEST ---")


--- STARTING ENGLISH TEST ---

Email: Hi, can we reschedule our meeting to next Monday? Let me know. Thanks.
--> AI Prediction: HAM

Email: CONGRATULATIONS! You have won a $1,000 Walmart Gift Card. Click here to claim it now for FREE!
--> AI Prediction: HAM

--- END OF TEST ---


**Ok**

In [43]:
# Test avec des mots-clés très agressifs
gros_spam = ["Viagra for sale! Send cash now to win money fast. Urgent business proposal."]
spam_count = vectorizer.transform(gros_spam)
print(f"Test Spam : {classifier.predict(spam_count)}")

Test Spam : ['SPAM']
